<a href="https://colab.research.google.com/github/EmilKarimov3/Emil_INFO4670_Fall2026/blob/main/Assignment2_Northgate_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# INFO 4670 / 4760 — Assignment 2 (Framework)
### Cleaning & Integrating the Northgate Data

Fill in each **# TODO** cell with your code, then run the **✅ Check** cell under it to see if it passes. Work top to bottom. When you're done, run the whole notebook once (Runtime → Run all), make sure it runs cleanly, and submit your **GitHub link**.

- Do every fix on a **copy** — never overwrite the raw files.
- The Week 5 Guided notebook shows every technique you need.
- You're graded on correct operations **and** justified decisions (see the rubric).

## Setup — load the three files (given)

In [22]:
import pandas as pd, numpy as np, os
try:
    students = pd.read_csv("student_records.csv")
except FileNotFoundError:
    from google.colab import files
    print("Upload student_records.csv, course_enrollments.csv, weekly_activity.csv")
    files.upload()
    students = pd.read_csv("student_records.csv")
enroll   = pd.read_csv("course_enrollments.csv")
activity = pd.read_csv("weekly_activity.csv")
# Golden rule: work on copies, never overwrite the raw files.
print("students", students.shape, "| enroll", enroll.shape, "| activity", activity.shape)

students (2027, 12) | enroll (8088, 4) | activity (32000, 4)


## Part A · Clean student_records

### A1 · Missing values
Find how many values are missing in `study_hours_reported` and store the count as **`n_missing_study`**. Then, in the markdown cell after your code, say in 1–2 sentences which of Han's methods you would use to handle it and why.
*Hint:* `.isna().sum()`

In [23]:
# TODO: set n_missing_study to the number of blank study_hours_reported values
# Keep an unchanged copy for later comparison.
students_raw = students.copy()

# Find every column with missing values.
missing_counts = students.isna().sum()
print(missing_counts[missing_counts > 0])

n_missing_study = int(students["study_hours_reported"].isna().sum())

print("Total records:", len(students))
print("Missing study hours:", n_missing_study)
print(students["study_hours_reported"].agg(["mean", "median", "skew"]))

study_hours_reported    255
dtype: int64
Total records: 2027
Missing study hours: 255
mean      10.486005
median    10.500000
skew      -0.066008
Name: study_hours_reported, dtype: float64


**Your justification (1–2 sentences):**  I would use Han’s method of mean substitution because the observed values for number of hours spent in studying are approximately symmetric (mean = 10.49, median = 10.50). The students with missing data will be retained in the dataset, but the use of mean substitution will reduce variability.

In [24]:
# ✅ Check
try:
    assert n_missing_study == 255
    print("✅ A1 correct — 255 missing (n = 1772 present)")
except Exception:
    print("❌ A1 not yet — set n_missing_study to the count of blank study_hours_reported")

✅ A1 correct — 255 missing (n = 1772 present)


### A2 · Inconsistent categories
Standardize the `housing` column into its three real groups and store the result as a new column **`students["housing_clean"]`**.
*Hint:* `.str.strip().str.lower().map({...})`

In [25]:
# TODO: create students["housing_clean"] with exactly 3 standardized groups
print("Housing counts BEFORE cleaning:")
print(students["housing"].value_counts(dropna=False))

housing_map = {
    "on-campus": "On-Campus",
    "off-campus": "Off-Campus",
    "off campus": "Off-Campus",
    "with family": "With Family"
}

students["housing_clean"] = (
    students["housing"]
    .str.strip()
    .str.lower()
    .map(housing_map)
)

# Make sure every original label was recognized.
assert students["housing_clean"].notna().all()

print("\nHousing counts AFTER cleaning:")
print(students["housing_clean"].value_counts())

Housing counts BEFORE cleaning:
housing
Off-Campus     755
On-Campus      480
With Family    420
off campus     113
Off-campus      77
with family     72
on-campus       69
 On-Campus      41
Name: count, dtype: int64

Housing counts AFTER cleaning:
housing_clean
Off-Campus     945
On-Campus      590
With Family    492
Name: count, dtype: int64


In [26]:
# ✅ Check
try:
    assert students["housing_clean"].nunique() == 3
    print("✅ A2 correct — 3 groups:", {k:int(v) for k,v in students["housing_clean"].value_counts().items()})
except Exception:
    print("❌ A2 not yet — housing_clean should have exactly 3 groups (expect 590 / 945 / 492)")

✅ A2 correct — 3 groups: {'Off-Campus': 945, 'On-Campus': 590, 'With Family': 492}


### A3 · Errors vs. extremes
Find the impossible values. Store the sorted unique impossible ages as **`impossible_ages`** and the number of rows with negative work hours as **`n_neg_work`**. (Remember: extreme-but-valid values like a long commute are *kept*.)
*Hint:* boolean masks on `age` and `work_hours_per_week`.

In [27]:
# TODO
# this is to flag impossible ages and ages unsuitable for this university roster.
age_error = (students["age"] < 0) | (students["age"] > 130)
child_age = students["age"].isin([0, 1, 3])
invalid_age = age_error | child_age

impossible_ages = sorted(
    students.loc[invalid_age, "age"].unique().tolist()
)

negative_work = students["work_hours_per_week"] < 0
n_neg_work = int(negative_work.sum())

print("Invalid ages:", impossible_ages)
print("Rows with negative work hours:", n_neg_work)

# this show the affected records.
print(students.loc[
    invalid_age | negative_work,
    ["student_id", "age", "work_hours_per_week"]
])

Invalid ages: [-22, 0, 1, 3, 199, 220]
Rows with negative work hours: 4
     student_id  age  work_hours_per_week
34    NU-101130    0                   12
420   NU-101030    1                   20
569   NU-105466   22                   -8
975   NU-102239  199                    0
1009  NU-108817   24                  -10
1263  NU-104748   24                   -5
1400  NU-103708  -22                   40
1533  NU-106800  220                   25
1534  NU-101873   23                  -12
1552  NU-103050    3                   10


In [28]:
# ✅ Check
try:
    assert 220 in impossible_ages and -22 in impossible_ages and n_neg_work == 4
    print("✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows")
except Exception:
    print("❌ A3 not yet — check ages (e.g. -22, 199, 220) and count negative work hours (expect 4)")

✅ A3 correct — impossible ages incl. -22/199/220; 4 negative work-hour rows


### A4 · Duplicates
Remove duplicate **student** records and store the result as **`students_dedup`**. Then, in the markdown cell after your code, explain in one sentence why you must NOT de-duplicate `enroll` or `activity` by ID.
*Hint:* `.drop_duplicates()` — think about exact vs. near-duplicates.

In [29]:
# TODO: build students_dedup (one row per student)
# to keep the first record for each student ID.
students_dedup = students.drop_duplicates(
    subset="student_id", keep="first"
).copy()

print("Rows before:", len(students))
print("Rows after:", len(students_dedup))
print("Duplicates removed:", len(students) - len(students_dedup))

# this replace invalid entries with valid-column medians.
# this flags identify these replacements as estimates.
for column, bad_values in [
    ("age", students_dedup["age"].isin(impossible_ages)),
    ("work_hours_per_week",
     students_dedup["work_hours_per_week"] < 0)
]:
    students_dedup[column + "_imputed"] = bad_values
    replacement = students_dedup.loc[~bad_values, column].median()
    students_dedup.loc[bad_values, column] = replacement

    print(column, "values replaced:", int(bad_values.sum()),
          "median used:", replacement)

# this fill missing study hours using the mean after de-duplication.
students_dedup["study_hours_imputed"] = (
    students_dedup["study_hours_reported"].isna()
)

study_mean = students_dedup["study_hours_reported"].mean()
students_dedup["study_hours_reported"] = (
    students_dedup["study_hours_reported"].fillna(study_mean)
)

print("Study-hour values filled:",
      int(students_dedup["study_hours_imputed"].sum()))
print("Study-hour mean used:", round(study_mean, 3))
print("Rows after filling:", len(students_dedup))

Rows before: 2027
Rows after: 2000
Duplicates removed: 27
age values replaced: 6 median used: 25.0
work_hours_per_week values replaced: 4 median used: 18.0
Study-hour values filled: 254
Study-hour mean used: 10.469
Rows after filling: 2000


**Why not de-dupe enroll / activity? (1 sentence):** I don't drop duplicate student ID's from activity or enrollments because each row is either an instance of a particular student enrolled in a course in a term or an instance of a particular student in a particular week.

In [30]:
# ✅ Check
try:
    assert len(students_dedup) == 2000 and students_dedup["student_id"].is_unique
    print("✅ A4 correct — 2000 unique students (from 2027 rows)")
except Exception:
    print("❌ A4 not yet — students_dedup should be 2000 rows, one per student")

✅ A4 correct — 2000 unique students (from 2027 rows)


## Part B · Integrate the three files

### B5 · Standardize the key & integrate
Build one **row-per-student** analysis table called **`analysis`**: start from `students_dedup`, add a standardized numeric key, and merge in a per-student summary of `activity` (e.g., total `minutes_active`).
*Hint:* make the key with `.str.replace("NU-","")` → `int`; summarize activity with `groupby(...).sum()`; then `merge`.

In [31]:
# TODO: build the standardized key and the one-row-per-student "analysis" table
# Start from the cleaned student copy.
analysis = students_dedup.copy()

# this create the same numeric student key in all three tables.
analysis["sid"] = (
    analysis["student_id"]
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

activity_copy = activity.copy()
activity_copy["sid"] = (
    activity_copy["student_id"]
    .str.replace("NU-", "", regex=False)
    .astype(int)
)

enroll_copy = enroll.copy()
enroll_copy["sid"] = pd.to_numeric(
    enroll_copy["sid"], errors="raise"
).astype(int)

# this summarize all weekly records for each student.
activity_summary = activity_copy.groupby("sid", as_index=False).agg(
    total_minutes_active=("minutes_active", "sum"),
    total_lms_clicks=("lms_clicks", "sum"),
    weeks_observed=("week", "nunique")
)

# this summarize course enrollments for each student.
course_summary = enroll_copy.groupby("sid", as_index=False).agg(
    enrollment_count=("course", "size"),
    unique_courses=("course", "nunique")
)

# is to left joins retain every student in the cleaned roster.
analysis = analysis.merge(
    course_summary, on="sid", how="left", validate="one_to_one"
)

analysis = analysis.merge(
    activity_summary, on="sid", how="left", validate="one_to_one"
)

print("Rows after joining:", len(analysis))
print("Unique students:", analysis["student_id"].nunique())

analysis[[
    "student_id", "sid", "enrollment_count", "total_minutes_active"
]].head()

Rows after joining: 2000
Unique students: 2000


,student_id,sid,enrollment_count,total_minutes_active
0,NU-101508,101508,5,2750
1,NU-102438,102438,4,2792
2,NU-102385,102385,5,5024
3,NU-100767,100767,3,2719
4,NU-106054,106054,4,2916


In [32]:
# ✅ Check
try:
    assert len(analysis) == 2000 and analysis["student_id"].is_unique
    print("✅ B5 correct — one row per student, 2000 rows")
except Exception:
    print("❌ B5 not yet — analysis should have one row per student (2000)")

✅ B5 correct — one row per student, 2000 rows


### B6 · Verify the join
Report how many `enroll` rows match a student in your standardized key. Store the count as **`matched`**.
*Hint:* `enroll["sid"].isin(set_of_keys).sum()`

In [33]:
# TODO
student_keys = set(analysis["sid"])

# this count enrollment matches and orphan IDs.
course_match = enroll_copy["sid"].isin(student_keys)
matched = int(course_match.sum())
orphan_ids = enroll_copy.loc[~course_match, "sid"].nunique()

print("Matched enrollment rows:", matched)
print("Unmatched enrollment rows:", int((~course_match).sum()))
print("Enrollment orphan IDs:", orphan_ids)

# this check activity matches too.
activity_match = activity_copy["sid"].isin(student_keys)

print("Matched activity rows:", int(activity_match.sum()))
print("Activity orphan IDs:",
      activity_copy.loc[~activity_match, "sid"].nunique())

# this confirm that integration preserved the cleaned roster.
print("Cleaned roster rows:", len(students_dedup))
print("Final analysis rows:", len(analysis))

assert len(analysis) == len(students_dedup)
assert analysis["student_id"].is_unique
assert set(analysis["student_id"]) == set(students_dedup["student_id"])

# this confirm that the joined summaries include all matched records.
assert analysis["enrollment_count"].sum() == matched
assert analysis["total_minutes_active"].sum() == (
    activity_copy.loc[activity_match, "minutes_active"].sum()
)

Matched enrollment rows: 8041
Unmatched enrollment rows: 47
Enrollment orphan IDs: 14
Matched activity rows: 32000
Activity orphan IDs: 0
Cleaned roster rows: 2000
Final analysis rows: 2000


In [34]:
# ✅ Check
try:
    assert matched == 8041
    print("✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)")
except Exception:
    print("❌ B6 not yet — count enrollment rows whose sid is in your student keys (expect 8041)")

✅ B6 correct — 8041 of 8088 enrollment rows match (14 orphan IDs)


## Part C · Transform

### C7 · Parse the dates
Parse `enrollment_date` so no valid date is lost. Store the parsed series as **`dates_parsed`** and check the number of NaT (blanks).
*Hint:* `pd.to_datetime(..., format="mixed", errors="coerce")` — compare NaT before and after.

In [35]:
# TODO
# this count blanks before parsing.
date_text = (
    analysis["enrollment_date"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

blanks_before = int(date_text.isna().sum())

# this recognize mixed formats. Slash dates use month/day/year.
dates_parsed = pd.to_datetime(
    date_text,
    format="mixed",
    errors="coerce",
    dayfirst=False
)

blanks_after = int(dates_parsed.isna().sum())
lost_dates = date_text.notna() & dates_parsed.isna()

print("Date blanks before:", blanks_before)
print("Date blanks after:", blanks_after)
print("Nonblank dates lost:", int(lost_dates.sum()))
print("Dates recognized:", int(dates_parsed.notna().sum()))

assert not lost_dates.any(), "Some nonblank dates failed to parse."

# this store the parsed dates in the final analysis table.
analysis["enrollment_date"] = dates_parsed

Date blanks before: 0
Date blanks after: 0
Nonblank dates lost: 0
Dates recognized: 2000


In [36]:
# ✅ Check
try:
    assert dates_parsed.isna().sum() == 0
    print("✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)")
except Exception:
    print("❌ C7 not yet — parse every format so no valid date becomes NaT")

✅ C7 correct — all dates parsed, 0 lost (a naive parse would lose ~1470)


### C8 · Normalize & discretize
Add two columns to `analysis`: a **z-scored** numeric column stored as **`analysis["study_z"]`**, and a **GPA band** column stored as **`analysis["gpa_band"]`** (bin `final_gpa` into 4 bands).
*Hint:* z-score = `(x - x.mean()) / x.std()`; bands = `pd.cut(..., bins=[-0.01,1,2,3,4])`.

In [37]:
# TODO: add analysis["study_z"] and analysis["gpa_band"]
# Z-score study hours: subtract the mean and divide by the standard deviation.
study = analysis["study_hours_reported"]
analysis["study_z"] = (study - study.mean()) / study.std()

# is to create four GPA bands, including GPA 0 in the first band.
analysis["gpa_band"] = pd.cut(
    analysis["final_gpa"],
    bins=[0, 1, 2, 3, 4],
    labels=["0–1", ">1–2", ">2–3", ">3–4"],
    include_lowest=True
)

# to confirm neither transformation introduced missing values.
assert analysis["study_z"].notna().all()
assert analysis["gpa_band"].notna().all()

print("Z-score mean:", round(analysis["study_z"].mean(), 6))
print("Z-score standard deviation:", round(analysis["study_z"].std(), 6))
print("\nStudents in each GPA band:")
print(analysis["gpa_band"].value_counts().sort_index())

Z-score mean: -0.0
Z-score standard deviation: 1.0

Students in each GPA band:
gpa_band
0–1       70
>1–2     528
>2–3    1102
>3–4     300
Name: count, dtype: int64


In [38]:
# ✅ Check
try:
    assert analysis["gpa_band"].nunique() == 4 and abs(analysis["study_z"].mean()) < 0.01
    print("✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added")
except Exception:
    print("❌ C8 not yet — add a z-scored column and a 4-band gpa_band column")

✅ C8 correct — z-score (mean ≈ 0) and 4 GPA bands added


## Part D · Deliver & reflect

### D9 · Write the clean file
Write your clean `analysis` table to **`northgate_clean.csv`** (do NOT overwrite the raw files).
*Hint:* `.to_csv("northgate_clean.csv", index=False)`

In [39]:
# TODO: write analysis to northgate_clean.csv
# is to save the clean table to a NEW file.
analysis.to_csv(
    "northgate_clean.csv",
    index=False,
    date_format="%Y-%m-%d"
)

print("File saved: northgate_clean.csv")
print("Student rows exported:", len(analysis))

File saved: northgate_clean.csv
Student rows exported: 2000


In [40]:
# ✅ Check
try:
    assert os.path.exists("northgate_clean.csv")
    print("✅ D9 correct — northgate_clean.csv written (raw files untouched)")
except Exception:
    print("❌ D9 not yet — write analysis to northgate_clean.csv")

✅ D9 correct — northgate_clean.csv written (raw files untouched)


### D10 · Cleaning log
In the markdown cell below, list each decision you made above and a one-line justification for it (missing values, housing, impossible values, duplicates, key, dates). *This is graded — no code needed.*



- Missing data: The study hours column was the only column to have missing data. There were 255 entries prior to removing duplicates and 254 after duplicates were removed. All other missing data was filled with the mean of the cleaned roster (approximately 10.47 hours). This was done because the study hours data were approximately symmetric. This filled all data and kept all 2,000 students but decreased variation in the data.

- Housing: Data were cleaned by normalizing the spelling, capitalization, and spacing to be either With Family, On-Campus or Off-Campus so that similar values would not be treated as distinct values.

- Invalid ages: Ages of −22, 0, 1, 3, 199 and 220 were filled with the median age of 25 for valid age data. These were either impossible or not realistic ages for university students. After filling these values, all other attributes of the students were retained.

- Invalid work hours: Four negative work hour values were replaced with the median value for valid work hours (18). Negative work hours are impossible. Using the median for this fill decreases the effect of outliers. All imputations are flagged.

- Outliers for valid data: Very long commute times were retained. An outlier may still be valid depending on the meaning of the variable. An invalid value would violate either the meaning or the constraints of the variable.

- Duplicates: 27 duplicate students were removed. This decreased the number of rows from 2,027 to 2,000 unique students. The first instance of a given student was retained as there was no timestamp to determine the most accurate entry.

- Duplicate student IDs across enrollments and activities: Duplicate student IDs across enrollments and activities were not removed. These are student-course-term and student-weeks respectively. Each row still represents an enrollment or activity.

- Keys and joining: Student IDs were normalized into keys. Student course enrollments and student activity per week were summarized by student. The cleaned roster was joined using left joins to retain the roster data. Joins were verified to ensure no duplicate students existed across joins.

- Join validation: There were 8,041 enrollment rows that matched and 47 rows that did not match. These 47 unmatched rows corresponded to 14 orphan student IDs. All 32,000 activity rows matched. The orphan enrollment IDs were not included in the roster join. The roster join still included 2,000 students.

- Dates: Various date formats were converted. Slash dates were converted to month/day/year format. Blank data before and after dates were also compared. No non-blank dates were lost during this process.

- Transformations: Study hours for non-missing values were normalized using a z-score. GPAs were divided into four categories: 0-1, &gt;1-2, &gt;2-3 and &gt;3-4. This normalizes the data and allows for easier comparisons of GPAs.

- Output: All data was operated on copies and the joined table was saved as northgate_clean.csv without an additional index column. The original files remained unchanged.List item




### D11 · Payoff
Using your clean `analysis` table, report the **mean GPA** and **one relationship** you find interesting, then note in one sentence how cleaning changed the picture versus the raw data.

# TODO: compute the mean GPA and explore one relationship on the CLEAN data


In [41]:
# ✅ Check
# this compare mean GPA before and after cleaning.
raw_mean_gpa = students_raw["final_gpa"].mean()
clean_mean_gpa = analysis["final_gpa"].mean()

print(f"Raw mean GPA: {raw_mean_gpa:.4f}")
print(f"Clean mean GPA: {clean_mean_gpa:.4f}")

# this explore the relationship between commute distance and GPA.
commute_gpa_corr = analysis["commute_miles"].corr(
    analysis["final_gpa"]
)

print(f"Commute–GPA correlation: {commute_gpa_corr:.3f}")
print("Students in the clean table:", len(analysis))

Raw mean GPA: 2.3075
Clean mean GPA: 2.3054
Commute–GPA correlation: -0.295
Students in the clean table: 2000


The data was cleaned and there were 2,000 students in the data with a mean GPA of 2.3054 (or approximately 2.31). There is a relationship between commute and GPA (−0.295). This implies students who live farther away tend to have lower GPAs. However, this does not necessarily imply that commuting leads to lower GPAs. Duplicates of students were removed from the data and this caused a small decrease in the mean GPA from 2.3075 to 2.3054. The number of students however was greatly improved through data cleaning with minimal impact on the overall GPA.